<a href="https://colab.research.google.com/github/rachmi00/Traffic-Sign-Recognition/blob/main/Traffic_Sign_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kagglehub

In [2]:
import kagglehub
path = kagglehub.dataset_download("ferrantealessandro/street-sign-set")
print(path)

100%|██████████| 622M/622M [00:05<00:00, 109MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/ferrantealessandro/street-sign-set/versions/3


In [9]:
import os
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.yaml'):
            print(os.path.join(root, file))

/root/.cache/kagglehub/datasets/ferrantealessandro/street-sign-set/versions/3/StreetSignSet/data.yaml


In [14]:
import yaml
# The file is located in 'StreetSignSet' rather than 'dataset'
with open(f"{path}/StreetSignSet/data.yaml") as f:
    ss_config = yaml.safe_load(f)
print(ss_config['names'])

ss_names = ss_config['names']
print(f"\nStreetSignSet has {len(ss_names)} classes.")

['prio_give_way', 'prio_stop', 'prio_priority_road', 'forb_speed_over_5', 'forb_speed_over_10', 'forb_speed_over_20', 'forb_speed_over_30', 'forb_speed_over_40', 'forb_speed_over_50', 'forb_speed_over_60', 'forb_speed_over_70', 'forb_speed_over_80', 'forb_speed_over_90', 'forb_speed_over_100', 'forb_speed_over_110', 'forb_speed_over_120', 'forb_speed_over_130', 'forb_no_entry', 'forb_no_parking', 'forb_no_stopping', 'forb_overtake_car', 'forb_overtake_trucks', 'forb_trucks', 'forb_turn_left', 'forb_turn_right', 'forb_weight_over_3.5t', 'forb_weight_over_7.5t', 'forb_u_turn', 'info_bus_station', 'info_crosswalk', 'info_highway', 'info_one_way', 'info_parking', 'info_taxi_parking', 'warn_children', 'warn_construction', 'warn_crosswalk', 'warn_cyclists', 'warn_left_curve', 'warn_right_curve', 'warn_domestic_animals', 'warn_other_dangers', 'warn_poor_road_surface', 'warn_roundabout', 'warn_sharp_left_curve', 'warn_sharp_right_curve', 'warn_slippery_road', 'warn_hump', 'warn_traffic_light',

In [15]:
STREETSIGN_TO_OUR_CLASS = {
    "forb_speed_over_30":  0,   # Speed_Limit_30
    "forb_speed_over_50":  1,   # Speed_Limit_50
    "prio_priority_road":  2,   # Priority_Road
    "prio_give_way":       3,   # Give_Way
    "prio_stop":           4,   # Stop
    "forb_no_entry":       5,   # No_Entry
    "warn_construction":   6,   # Road_Work
    "warn_traffic_light":  7,   # Traffic_Lights_Ahead
    "warn_crosswalk":      8,   # Pedestrian_Crossing
    "warn_roundabout":     9,   # Roundabout
}

OUR_CLASS_NAMES = [
    "Speed_Limit_30", "Speed_Limit_50", "Priority_Road", "Give_Way", "Stop",
    "No_Entry", "Road_Work", "Traffic_Lights_Ahead", "Pedestrian_Crossing", "Roundabout",
]

In [16]:
#Build StreetSignSet's own internal index → our class ID
# (StreetSignSet's data.yaml gives us names in order: index 0 = ss_names[0], etc.)
ss_index_to_our_class = {}
for ss_index, ss_name in enumerate(ss_names):
    if ss_name in STREETSIGN_TO_OUR_CLASS:
        ss_index_to_our_class[ss_index] = STREETSIGN_TO_OUR_CLASS[ss_name]

print(f"\nMatched {len(ss_index_to_our_class)} of {len(STREETSIGN_TO_OUR_CLASS)} target classes:")
for ss_index, our_id in ss_index_to_our_class.items():
    print(f"  StreetSignSet[{ss_index}] '{ss_names[ss_index]}'  →  our class {our_id} ({OUR_CLASS_NAMES[our_id]})")

missing = set(STREETSIGN_TO_OUR_CLASS.keys()) - {ss_names[i] for i in ss_index_to_our_class}
if missing:
    print(f"\n[!] WARNING — these expected classes were NOT found in StreetSignSet's data.yaml: {missing}")
    print("    Check for spelling differences and adjust STREETSIGN_TO_OUR_CLASS above.")


Matched 10 of 10 target classes:
  StreetSignSet[0] 'prio_give_way'  →  our class 3 (Give_Way)
  StreetSignSet[1] 'prio_stop'  →  our class 4 (Stop)
  StreetSignSet[2] 'prio_priority_road'  →  our class 2 (Priority_Road)
  StreetSignSet[6] 'forb_speed_over_30'  →  our class 0 (Speed_Limit_30)
  StreetSignSet[8] 'forb_speed_over_50'  →  our class 1 (Speed_Limit_50)
  StreetSignSet[17] 'forb_no_entry'  →  our class 5 (No_Entry)
  StreetSignSet[35] 'warn_construction'  →  our class 6 (Road_Work)
  StreetSignSet[36] 'warn_crosswalk'  →  our class 8 (Pedestrian_Crossing)
  StreetSignSet[43] 'warn_roundabout'  →  our class 9 (Roundabout)
  StreetSignSet[48] 'warn_traffic_light'  →  our class 7 (Traffic_Lights_Ahead)


In [20]:
import shutil
import random
from pathlib import Path
from collections import Counter

DATASET_DIR = Path("/content/dataset")
TRAIN_DIR = DATASET_DIR / "train"
VALID_DIR = DATASET_DIR / "valid"

OUR_CLASS_NAMES = [
    "Speed_Limit_30", "Speed_Limit_50", "Priority_Road", "Give_Way", "Stop",
    "No_Entry", "Road_Work", "Traffic_Lights_Ahead", "Pedestrian_Crossing", "Roundabout",
]

VAL_FRACTION = 0.20   # 80% train, 20% valid
RANDOM_SEED = 42      # fixed seed = reproducible split, defensible in Chapter 4


In [21]:

# ── Step 1: Pool every existing image+label pair into one list ──────────────
all_pairs = []   # list of (image_path, label_path) tuples

for split_dir in (TRAIN_DIR, VALID_DIR):
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"
    if not img_dir.exists():
        continue
    for img_path in img_dir.iterdir():
        if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
            continue
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            all_pairs.append((img_path, lbl_path))

print(f"Pooled {len(all_pairs)} image+label pairs from existing dataset.")



Pooled 1675 image+label pairs from existing dataset.


In [18]:

import shutil
from pathlib import Path

DATASET_DIR = Path("/content/dataset")
DATASET_DIR.mkdir(parents=True, exist_ok=True)   # do NOT delete if it already
                                                   # has GTSDB data from a prior run

# Fix: use the 'path' variable defined in the download cell
SS_DIR = Path(path) / "StreetSignSet"

copied_count = {"train": 0, "valid": 0}

# StreetSignSet uses 'val' and 'test' — we fold both into our 'train'/'valid'
# scheme. test → valid, so our final validation set draws from StreetSignSet's
# original val AND test splits combined.
SPLIT_MAP = {"train": "train", "val": "valid", "test": "valid"}

for ss_split, our_split in SPLIT_MAP.items():
    ss_img_dir = SS_DIR / ss_split / "images"
    ss_lbl_dir = SS_DIR / ss_split / "labels"

    if not ss_img_dir.exists():
        print(f"  [skip] {ss_split} — folder not found in StreetSignSet download")
        continue

    out_img_dir = DATASET_DIR / our_split / "images"
    out_lbl_dir = DATASET_DIR / our_split / "labels"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    for lbl_path in ss_lbl_dir.glob("*.txt"):
        kept_lines = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                ss_class_id = int(parts[0])
                if ss_class_id in ss_index_to_our_class:
                    our_class_id = ss_index_to_our_class[ss_class_id]
                    kept_lines.append(f"{our_class_id} {' '.join(parts[1:])}")

        if not kept_lines:
            continue   # this image has none of our 10 target classes — skip it

        img_stem = lbl_path.stem
        img_path = None
        for ext in (".jpg", ".jpeg", ".png"):
            candidate = ss_img_dir / f"{img_stem}{ext}"
            if candidate.exists():
                img_path = candidate
                break
        if img_path is None:
            continue

        dst_img = out_img_dir / f"ss_{img_path.name}"
        dst_lbl = out_lbl_dir / f"ss_{img_stem}.txt"

        shutil.copy2(img_path, dst_img)
        dst_lbl.write_text("\n".join(kept_lines))

        copied_count[our_split] += 1

print(f"\n── Build complete ──────────────────────────────────────")
print(f"  StreetSignSet images in train : {copied_count['train']}")
print(f"  StreetSignSet images in valid : {copied_count['valid']}")


  [skip] val — folder not found in StreetSignSet download

── Build complete ──────────────────────────────────────
  StreetSignSet images in train : 1611
  StreetSignSet images in valid : 64


In [19]:
from collections import Counter

def count_instances(split):
    label_dir = DATASET_DIR / split / 'labels'
    counter = Counter()
    if not label_dir.exists():
        return counter
    for lbl_file in label_dir.glob('*.txt'):
        with open(lbl_file, 'r') as f:
            for line in f:
                parts = line.split()
                if parts:
                    class_id = int(parts[0])
                    counter[class_id] += 1
    return counter

print("Per-class instance counts:")
for split in ['train', 'valid']:
    counts = count_instances(split)
    print(f"\n--- {split.upper()} ---")
    for i, name in enumerate(OUR_CLASS_NAMES):
        print(f"{i} {name}: {counts[i]} instances")

Per-class instance counts:

--- TRAIN ---
0 Speed_Limit_30: 302 instances
1 Speed_Limit_50: 242 instances
2 Priority_Road: 133 instances
3 Give_Way: 423 instances
4 Stop: 254 instances
5 No_Entry: 236 instances
6 Road_Work: 109 instances
7 Traffic_Lights_Ahead: 72 instances
8 Pedestrian_Crossing: 157 instances
9 Roundabout: 117 instances

--- VALID ---
0 Speed_Limit_30: 13 instances
1 Speed_Limit_50: 24 instances
2 Priority_Road: 5 instances
3 Give_Way: 11 instances
4 Stop: 4 instances
5 No_Entry: 6 instances
6 Road_Work: 3 instances
7 Traffic_Lights_Ahead: 0 instances
8 Pedestrian_Crossing: 4 instances
9 Roundabout: 3 instances
